In [0]:
%run ./00_import


In [0]:
%run ./01_env

In [0]:
%run ./02_load_data

In [0]:
%run ./03_transform_data

### 1. Détails : Production Line Deployment

In [0]:
# Base batch non supprimés
batches_active = batches.filter(F.col("deleted") == F.lit(False))

# Base sites avec batchs
sites_batches = (
    pl.alias("pl")
    .join(
        batches_active.alias("b"),
        F.col("pl.id_plant_production_line") == F.col("b.production_line"),
        "inner"
    )
)

kpi_data_digitalization = (
    sites_batches
    .groupBy(
        F.col("pl.id_plant_production_line"),
        F.col("pl.name")
    )
    .agg(F.count("b.id_batch").alias("batch_count"))
    .filter(F.col("batch_count") > 180)
    .agg(F.count("*").alias("kpi_value"))
    .withColumn("kpi_code", F.lit("DATA_DIGITALIZATION"))
)

kpi_metamodels = (
    sites_batches
    .filter(F.col("pl.common_code") != "PI6")
    .select("pl.name")
    .distinct()
    .agg(F.count("*").alias("kpi_value"))
    .withColumn("kpi_code", F.lit("METAMODELS"))
)

ai_sites = ["RO1", "PO1", "NG2", "PR1", "ST2", "NG1", "BU1"]

kpi_ai_recommendations = (
    sites_batches
    .filter(F.col("pl.common_code").isin(ai_sites))
    .select("pl.name")
    .distinct()
    .agg(F.count("*").alias("kpi_value"))
    .withColumn("kpi_code", F.lit("AI_RECOMMENDATIONS"))
)

trs_metadata_mapped = (
    trs_metadata
    .withColumn(
        "common_code",
        F.when(F.col("prd_line") == "burton1", "BT1")
         .when(F.col("prd_line") == "rouen1", "RO1")
         .when(F.col("prd_line") == "polisy1", "PO1")
         .when(F.col("prd_line") == "bolelemi1", "BO1")
         .when(F.col("prd_line") == "strasbourg1", "ST1")
         .when(F.col("prd_line") == "strasbourg2", "ST2")
         .when(F.col("prd_line") == "nogent1", "NG1")
         .when(F.col("prd_line") == "nogent2", "NG2")
         .when(F.col("prd_line") == "NYMBURK1", "NY1")
         .when(F.col("prd_line") == "prouvy1", "PR1")
         .when(F.col("prd_line") == "buzau1", "BU1")
         .when(F.col("prd_line") == "burton2", "BT2")
         .when(F.col("prd_line") == "HODONICE1", "HO1")
         .when(F.col("prd_line") == "KROMERIZ1", "KR1")
         .otherwise(F.col("prd_line"))
    )
)

kpi_time_tracking_production_notes = (
    pl.alias("pl")
    .join(batches_active.alias("b"), F.col("pl.id_plant_production_line") == F.col("b.production_line"), "inner")
    .join(notes.alias("bn"), F.col("b.id_batch") == F.col("bn.batch"), "inner")
    .join(trs_metadata_mapped.alias("md"), F.col("md.common_code") == F.col("pl.common_code"), "inner")
    .agg(F.countDistinct("pl.id_plant_production_line").alias("kpi_value"))
    .withColumn("kpi_code", F.lit("TIME_TRACKING_PRODUCTION_NOTES"))
)



### Détails : Data Quality & Completeness

In [0]:
#Total recommandations
reco_total_current = monitoring_current.agg(
    F.sum(
        F.coalesce(F.col("flg_echec"), F.lit(0)) +
        F.coalesce(F.col("flg_succes"), F.lit(0))
    ).alias("reco_total")
).collect()[0]["reco_total"]

#Uncaptured Aveva data
kpi_uncaptured_aveva_data = monitoring_current.filter(
    (F.col("succes_echec") == "ECHEC") &
    (F.col("flg_auto") == 1) &
    (F.col("Id_alerting") == 1)
).agg(
    F.count("flg_echec").alias("numerator")
).withColumn(
    "denominator", F.lit(reco_total_current)
).withColumn(
    "kpi_value", F.col("numerator") / F.col("denominator")
)

#Missing manual entries V1 
kpi_missing_manual_entries_1 = monitoring_current.filter(
    (F.col("flg_echec") == 1) &
    (F.col("has_manual_value") == 1) &
    (F.col("has_auto_value") == 0)
).agg(
    F.sum("flg_echec").alias("numerator")
).withColumn(
    "denominator", F.lit(reco_total_current)
).withColumn(
    "kpi_value", F.col("numerator") / F.col("denominator")
)

#Missing auto + manual entries V2
kpi_missing_auto_manual_entries = monitoring_current.filter(
    (F.col("flg_echec") == 1) &
    (F.col("has_manual_value") == 1) &
    (F.col("has_auto_value") == 1)
).agg(
    F.sum("flg_echec").alias("numerator")
).withColumn(
    "denominator", F.lit(reco_total_current)
).withColumn(
    "kpi_value", F.col("numerator") / F.col("denominator")
)

#Missing manual entries KPI PBI
kpi_missing_manual_entries = kpi_missing_manual_entries_1.withColumn(
    "numerator",
    F.col("numerator") + kpi_missing_auto_manual_entries.first()["numerator"]
).withColumn(
    "kpi_value",
    F.col("numerator") / F.col("denominator")
)

#% Batches with missing SAP mandatory data 
#Colonne is_missing_quality = 1 si la valeur de qualité est manquante
df_quality_check = (
    df_quality_results_long
    .withColumn(
        "is_missing_quality",
        F.when(F.col("quality_value").isNull(), 1).otherwise(0)
    )
)

df_quality_check_batch = (
    df_batch_completed
    .join(
        df_quality_check,
        "batch_id",
        "inner"
    )
)


kpi_batches_missing_sap_mandatory_data = (
    df_quality_check_batch
    .groupBy("batch_id")
    .agg(
        F.max("is_missing_quality").alias("has_missing_quality")
    )
    .filter(F.col("has_missing_quality") == 1)
    .agg(
        F.countDistinct("batch_id").alias("kpi_value")
    )
)

